# SSFL Colab GPU verification

Clones the private repo (already contains the built `cache/` dataset — no need to re-upload the 7.6 GB raw data) and runs the same smoke/verification check used locally, on whatever GPU this runtime has (T4/L4/A100).

**Before running:** Runtime > Change runtime type > select a GPU (T4/L4/A100). You'll also need a GitHub personal access token with `repo` scope (Settings > Developer settings > Personal access tokens) since the repo is private.

In [ ]:
!nvidia-smi

In [ ]:
import os
from getpass import getpass

REPO = "crAK1644/SSFL-with-Psuedo-Labeling"
REPO_DIR = "/content/SSFL-with-Psuedo-Labeling"

if not os.path.isdir(REPO_DIR):
    token = getpass("GitHub personal access token (repo scope): ")
    !git clone -q https://{token}@github.com/{REPO}.git {REPO_DIR}
    del token
else:
    print(f"{REPO_DIR} already exists, skipping clone")

%cd {REPO_DIR}

In [ ]:
REPO_DIR = "/content/SSFL-with-Psuedo-Labeling"

import torch
print("torch:", torch.__version__, "| cuda available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU runtime — Runtime > Change runtime type > GPU"
!pip install -q numpy pandas matplotlib

In [ ]:
import os
import sys

REPO_DIR = "/content/SSFL-with-Psuedo-Labeling"

# absolute path — survives runtime restarts / out-of-order cell runs,
# unlike a relative "src" path which silently breaks if cwd drifts
sys.path.insert(0, os.path.join(REPO_DIR, "src"))

from ssfl.data import load_client, load_open, load_test
from ssfl.models import build_model, resolve_device
from ssfl.config import derive_seed
from ssfl.methods import fl_logic, fd_logic, dsfl_logic, ssfl_logic

DEVICE = "auto"
print("resolved device:", resolve_device(DEVICE))

## Run 1: quick 4-method smoke check (mirrors the local run — 5 rounds, 5 clients)

Confirms the GPU path works end-to-end (no crashes/NaNs) before committing to a longer run.

In [ ]:
import time
import numpy as np

SCENARIO = 1
N_CLIENTS = 5
ROUNDS = 5
LOCAL_EPOCHS = 5
BATCH = 80
LR = 1e-4
SEED = 0
NUM_CLASSES = 11

clients_data = [load_client(SCENARIO, k) for k in range(N_CLIENTS)]
open_X = load_open()
X_test, y_test = load_test()
print(f"clients: {N_CLIENTS}, sizes: {[len(y) for _, y in clients_data]}")
print(f"open_X: {open_X.shape}, test: {X_test.shape}")

def model_fn():
    return build_model("cnn", num_classes=NUM_CLASSES)

def disc_fn():
    return build_model("cnn", num_classes=2)

print("\n=== FL (FedAvg) ===")
t0 = time.time()
weights = fl_logic.init_weights(model_fn, seed=SEED)
for r in range(1, ROUNDS + 1):
    weights, info = fl_logic.run_round(
        model_fn, weights, clients_data,
        round_num=r, run_seed=SEED, epochs=LOCAL_EPOCHS, batch=BATCH, lr=LR, device=DEVICE,
    )
    acc = fl_logic.evaluate(model_fn, weights, X_test, y_test, device=DEVICE)
    print(f"  round {r}: train_loss={info['train_loss']:.4f} test_acc={acc:.4f}")
print(f"FL done in {time.time()-t0:.1f}s")

In [ ]:
print("=== SSFL (default) ===")
t0 = time.time()
client_states = [(model_fn(), disc_fn()) for _ in range(N_CLIENTS)]
for k, (clf, disc) in enumerate(client_states):
    torch.manual_seed(derive_seed(SEED, client_id=k, round_num=0))
server_model = model_fn()
torch.manual_seed(derive_seed(SEED, client_id=N_CLIENTS, round_num=0))
global_labels = None
for r in range(1, ROUNDS + 1):
    global_labels, diag = ssfl_logic.run_round(
        client_states, server_model, clients_data, open_X, global_labels,
        round_num=r, run_seed=SEED, num_classes=NUM_CLASSES,
        lr=LR, batch=BATCH, local_epochs=LOCAL_EPOCHS, device=DEVICE,
    )
    acc = ssfl_logic.evaluate(server_model, X_test, y_test, device=DEVICE)
    print(f"  round {r}: zero_vote={diag['zero_vote']:5d}/{len(open_X)} server_acc={acc:.4f}")
print(f"SSFL done in {time.time()-t0:.1f}s")

## Run 2 (optional, once Run 1 looks healthy): longer SSFL bootstrap check

Locally (CPU/MPS), 8 clients x 20 rounds took ~300s and showed a cold start through round ~6 then a climb to ~50-55% test accuracy by round 18-20. Compare the timing and the round-7 takeoff point here against that baseline to gauge the GPU speedup — for this workload (small model, batch=80, sequential per-client loop) don't expect a dramatic speedup; the bottleneck is likely per-call overhead, not FLOPs.

In [ ]:
N_CLIENTS_LONG = 8
ROUNDS_LONG = 20

clients_data_long = [load_client(SCENARIO, k) for k in range(N_CLIENTS_LONG)]

t0 = time.time()
client_states = [(model_fn(), disc_fn()) for _ in range(N_CLIENTS_LONG)]
for k, (clf, disc) in enumerate(client_states):
    torch.manual_seed(derive_seed(SEED, client_id=k, round_num=0))
server_model = model_fn()
torch.manual_seed(derive_seed(SEED, client_id=N_CLIENTS_LONG, round_num=0))
global_labels = None
for r in range(1, ROUNDS_LONG + 1):
    global_labels, diag = ssfl_logic.run_round(
        client_states, server_model, clients_data_long, open_X, global_labels,
        round_num=r, run_seed=SEED, num_classes=NUM_CLASSES,
        lr=LR, batch=BATCH, local_epochs=LOCAL_EPOCHS, device=DEVICE,
    )
    acc = ssfl_logic.evaluate(server_model, X_test, y_test, device=DEVICE)
    print(f"  round {r:2d}: zero_vote={diag['zero_vote']:5d}/{len(open_X)} "
          f"vote_agree={diag['vote_agreement']:.3f} server_acc={acc:.4f} "
          f"server_trained_on={diag['server_trained_on']}")
print(f"done in {time.time()-t0:.1f}s")